# Module 05 · Unit 02
# Shortest Path and Attack Graphs

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Attack Graphs & Reachability \[AG\] |
| **Refresher** | Module 05 · Unit 01 — Graph Traversal and Reachability |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define the **weighted shortest path problem** formally
2. Trace Dijkstra's algorithm step-by-step on a weighted graph, maintaining the priority queue
3. Explain why Dijkstra requires non-negative edge weights
4. Model an enterprise network with **CVSS-based exploit costs** as a weighted attack graph
5. Compute the **minimum-cost attack path** from any entry point to any target
6. Interpret path cost in terms of attacker effort and defender detection windows


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $w(u,v)$ | Edge weight | "weight of arc $(u,v)$" | Cost of traversing the arc (e.g., exploit difficulty) |
| $\text{cost}(P)$ | Path cost | "cost of path $P$" | Sum of edge weights along path $P$ |
| $\delta_w(s,t)$ | Weighted shortest path | "minimum cost from $s$ to $t$" | Minimum $\sum w$ over all paths $s \rightsquigarrow t$ |
| $\text{dist}[v]$ | Tentative distance | "current best distance to $v$" | Running estimate in Dijkstra; equals $\delta_w(s,v)$ at termination |
| $\text{PQ}$ | Priority queue | "priority queue" | Min-heap ordered by tentative distance |
| $\text{CVSS}$ | Common Vulnerability Scoring System | "CVSS score" | 0.0–10.0 severity score for a vulnerability |

> **The upgrade from Unit 01.** BFS finds the path with the fewest hops.
> Dijkstra finds the path with the lowest **total weight** — the cheapest cost.
> In an attack graph where edge weights represent exploit difficulty, Dijkstra
> finds the path of least resistance. The attacker follows this path;
> the defender must raise its cost above the attacker's threshold.

---


## 1 · The Weighted Shortest Path Problem

Given a weighted directed graph $G = (V, E, w)$ where $w: E \to \mathbb{R}_{\geq 0}$
assigns a non-negative cost to each arc, and a source vertex $s \in V$, the
**single-source shortest path problem** asks:

$$\delta_w(s, t) = \min_{P: s \rightsquigarrow t} \sum_{(u,v) \in P} w(u,v)$$

for all $t \in V$ — find the minimum-cost path from $s$ to every reachable vertex.

### Why Weights Matter for Attack Graphs

BFS (Unit 01) treats every hop as equal cost. In reality, traversing different
edges in a network has very different difficulty:

- Exploiting a critical RCE vulnerability (CVSS 9.8) is easy — low attacker effort
- Exploiting a local privilege escalation (CVSS 4.3) requires prior access and more skill
- A trusted administrative connection might cost almost nothing to traverse

We model edge weight as the **attacker's cost** to traverse that connection —
proportional to exploit difficulty, not hop count. Low weight = easy to exploit.

### The Attacker's Objective

The attacker minimises total path cost:

$$\text{attack path}^* = \arg\min_{P: s \rightsquigarrow t} \text{cost}(P)$$

This is the path of **least resistance** — the route an attacker would rationally
take to minimise effort and detection risk. Computing it gives the defender a
precise target: if this path can be made prohibitively expensive (Unit 03), the
network is effectively defended against opportunistic attacks.

### Converting CVSS to Edge Weight

CVSS scores run from 0.0 (low severity) to 10.0 (critical). We want edge weight
to represent attacker effort — the *inverse* of severity:

$$w(u,v) = 10.0 - \text{CVSS}(u,v) + 0.1$$

A CVSS 9.8 vulnerability (easy to exploit) gets weight $\approx 0.3$ — cheap for
the attacker. A CVSS 2.1 vulnerability (hard to exploit) gets weight $\approx 8.0$
— expensive. The constant $0.1$ ensures no zero-weight edges, which can cause
algorithm instability.


## 2 · Dijkstra's Algorithm

**Dijkstra's algorithm** solves the single-source shortest path problem on graphs
with non-negative edge weights. It uses a **priority queue** (min-heap) ordered
by tentative distance.

### Algorithm

```
Dijkstra(G, s):
  dist[s] = 0;  dist[v] = ∞ for all v ≠ s
  parent[v] = None for all v
  PQ ← min-priority queue containing all vertices, keyed by dist
  
  while PQ is not empty:
    u ← extract-min(PQ)          ← vertex with smallest tentative distance
    for each arc (u, v) with weight w(u,v):
      if dist[u] + w(u,v) < dist[v]:
        dist[v]   = dist[u] + w(u,v)   ← relaxation step
        parent[v] = u
        decrease-key(PQ, v, dist[v])
```

**Time complexity:** $O((|V| + |E|) \log |V|)$ with a binary heap.

### The Relaxation Step

The core operation is **relaxation**: for each arc $(u,v)$, check whether
going through $u$ gives a cheaper path to $v$ than the current best:

$$\text{if } \text{dist}[u] + w(u,v) < \text{dist}[v]: \quad \text{update } \text{dist}[v]$$

Dijkstra processes vertices in order of their final shortest distance — each
vertex is "finalised" (popped from the priority queue) at most once, and once
finalised, its distance is provably optimal.

### Why Non-Negative Weights Are Required

If $w(u,v) < 0$, we could have $\text{dist}[u] + w(u,v) < \text{dist}[v]$
*after* $v$ has been finalised — violating the algorithm's correctness invariant.
For attack graphs with CVSS-based weights, all weights are positive, so
Dijkstra is always valid.

### Correctness Invariant

At each step, when vertex $u$ is extracted from the priority queue:

$$\text{dist}[u] = \delta_w(s, u)$$

*The tentative distance is the true shortest distance.* This holds because
all edge weights are non-negative — no later update can produce a shorter path
to $u$ than what was already found.


## 3 · The Enterprise Attack Graph Model

We model the enterprise network as a **weighted directed attack graph** where:

- **Vertices** represent hosts, services, and network zones
- **Arcs** represent exploitable connections
- **Edge weight** = $10.0 - \text{CVSS} + 0.1$ (attacker cost; lower = easier)

### CVSS Score Assignments

Each arc is labelled with both the CVSS score (severity) and the derived
attacker cost (weight). High-CVSS vulnerabilities have low weights — they are
the "easy" edges the attacker prefers to traverse.

### The Three Questions This Model Answers

1. **What is the minimum-cost attack path from any entry to any target?**
   → Dijkstra from the entry point.

2. **Which edges lie on the minimum-cost path?**
   → The path extracted from the parent pointers — these are the arcs the
   defender must focus on patching or monitoring.

3. **How much does patching a specific vulnerability raise the attack cost?**
   → Remove or increase the weight of that edge and re-run Dijkstra.
   The cost difference is the security value of the patch.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AG\]

| Dijkstra concept | Attack graph meaning |
|---|---|
| **Edge weight $w(u,v)$** | Attacker effort to exploit the connection |
| **$\delta_w(s,t)$** | Minimum attacker effort to reach $t$ from $s$ |
| **Relaxation** | "Is there a cheaper route to $v$ via $u$?" |
| **Minimum-cost path** | The attacker's preferred route — least resistance |
| **High-weight edge** | Hard-to-exploit link — effective barrier |
| **Low-weight edge** | Easy-to-exploit link — priority patching target |
| **$\delta_w(s,t) = \infty$** | Target unreachable — lateral movement impossible |
| **Increasing $w(e)$** | Patching or hardening a vulnerability |
| **Path edge subset** | The arcs on the critical path — top defender priorities |

**The defender's objective.** The attacker minimises cost; the defender maximises
the minimum cost (raises the floor). The formal statement:

$$\text{Secure} \;\equiv\; \delta_w(s, t) \geq \text{THRESHOLD} \quad \forall s \in \text{EntryPoints},\; t \in \text{Targets}$$

If the minimum attacker cost exceeds the attacker's budget or patience, the network
is effectively defended against that attacker class. Unit 03 will formalise exactly
how removing edges (zero-trust segmentation) makes $\delta_w(s,t) = \infty$.

---


## Python: Visualization & Verification

**1 · Dijkstra from Scratch** — implement the full algorithm with a priority queue,
logging every relaxation step so the process is completely transparent.

**2 · CVSS-Weighted Attack Graph** — build the full enterprise attack graph with
CVSS scores on every edge; run Dijkstra from multiple entry points; display the
minimum-cost paths to all targets.

**3 · Patch Impact Analysis** — compute how increasing specific edge weights
(patching vulnerabilities) changes the minimum attack cost; rank vulnerabilities
by the security value of patching them.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import heapq

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

def cvss_to_weight(cvss):
    """Convert CVSS severity to attacker cost. High CVSS = low cost = easy."""
    return round(10.0 - cvss + 0.1, 2)

print('Setup complete.')


### 1 · Dijkstra's Algorithm — Step-by-Step

We implement Dijkstra from scratch with full logging of every priority queue
operation and relaxation, then apply it to a small weighted graph so the
algorithm's progression is completely visible before we scale up.


In [ ]:
# ── 1 · Dijkstra from Scratch ────────────────────────────────────────────────

def dijkstra(G, source, weight_attr='weight', verbose=False):
    """
    Dijkstra's algorithm.
    Returns (dist, parent) dicts.
    G must be a weighted DiGraph with edge attribute weight_attr.
    """
    INF  = float('inf')
    dist = {v: INF for v in G.nodes()}
    par  = {v: None for v in G.nodes()}
    dist[source] = 0.0

    # Priority queue: (tentative_dist, vertex)
    pq = [(0.0, source)]
    finalised = set()
    steps = []

    while pq:
        d_u, u = heapq.heappop(pq)
        if u in finalised:
            continue
        finalised.add(u)

        for v in G.successors(u):
            w   = G[u][v].get(weight_attr, 1)
            new = d_u + w
            if new < dist[v]:
                dist[v] = new
                par[v]  = u
                heapq.heappush(pq, (new, v))
                steps.append((u, v, w, new))
                if verbose:
                    print(f"  Relax ({u}→{v}): {d_u:.2f} + {w:.2f} = {new:.2f}  "
                          f"[prev dist={dist[v]:.2f if dist[v]<float('inf') else '∞'}]")

    return dist, par, steps

def reconstruct_path(parent, source, target):
    """Reconstruct path from parent pointers."""
    path, v = [], target
    while v is not None:
        path.append(v); v = parent[v]
    path.reverse()
    return path if path[0] == source else []

# ── Small worked example ──────────────────────────────────────────────────────
G_small = nx.DiGraph()
edges_small = [
    ('A', 'B', 4), ('A', 'C', 2), ('C', 'B', 1),
    ('B', 'D', 5), ('C', 'D', 8), ('C', 'E', 10),
    ('B', 'E', 2), ('D', 'E', 2), ('E', 'F', 3), ('D', 'F', 6),
]
for u, v, w in edges_small:
    G_small.add_edge(u, v, weight=w)

print("Dijkstra trace from 'A':")
dist_s, par_s, steps_s = dijkstra(G_small, 'A', verbose=True)

print(f"\nFinal distances from A:")
for v in sorted(dist_s):
    path = reconstruct_path(par_s, 'A', v)
    cost = dist_s[v]
    print(f"  A → {v}: cost={cost:.1f}  path={' → '.join(path)}")

# ── Visualise with distance labels ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
pos_s = {'A':(0,1), 'B':(2,2), 'C':(2,0), 'D':(4,2), 'E':(4,0), 'F':(6,1)}

edge_labels = {(u,v): f"w={w}" for u,v,w in edges_small}

# Highlight shortest path A→F
path_af  = reconstruct_path(par_s, 'A', 'F')
path_set = set(zip(path_af, path_af[1:]))

for u,v,w in edges_small:
    color = TS_RED if (u,v) in path_set else TS_GRAY
    lw    = 3.0   if (u,v) in path_set else 1.2
    alpha = 0.95  if (u,v) in path_set else 0.35
    ax.annotate('', xy=pos_s[v], xytext=pos_s[u],
                arrowprops=dict(arrowstyle='->', color=color, lw=lw, alpha=alpha))
    mx, my = (pos_s[u][0]+pos_s[v][0])/2, (pos_s[u][1]+pos_s[v][1])/2
    ax.text(mx, my+0.12, str(w), ha='center', fontsize=9, color=color, fontweight='bold')

node_labels = {v: f"{v}
d={dist_s[v]:.0f}" for v in G_small.nodes()}
nx.draw_networkx_nodes(G_small, pos_s, ax=ax, node_color=TS_BLUE,
                       node_size=900, alpha=0.9)
for v, (x,y) in pos_s.items():
    ax.text(x, y+0.05, v, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(x, y-0.25, f"d={dist_s[v]:.0f}", ha='center', fontsize=8, color=TS_GRAY)

ax.set_title(f"Dijkstra's Algorithm — Shortest Paths from A\n"
             f"Red path = minimum cost A→F  (cost={dist_s['F']:.0f})",
             pad=10, fontsize=11)
ax.set_xlim(-0.5, 7); ax.set_ylim(-0.5, 2.8)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The relaxation trace shows Dijkstra discovering cheaper routes as it processes
vertices in order of their current best distance. The key moment is when it
discovers that $A \to C \to B$ (cost $2 + 1 = 3$) is cheaper than the direct
$A \to B$ (cost $4$) — the relaxation step updates $\text{dist}[B]$ from 4 to 3.

The final distances show the minimum-cost path to every vertex. The minimum
cost path to $F$ is $A \to C \to B \to E \to F$ with total cost 8 — not the
fewest hops (there are 2-hop and 3-hop alternatives) but the cheapest sum of
weights.

This is the key difference from BFS: **minimum cost ≠ minimum hops**. An attacker
with a specific budget always prefers the minimum-cost path, even if it takes more
steps.


### 2 · CVSS-Weighted Enterprise Attack Graph

We build the full enterprise attack graph with CVSS scores on every edge,
convert to attacker costs, and run Dijkstra from every entry point to every
target. The output is a complete attack cost matrix — every (entry, target) pair
with its minimum attacker cost and the specific path.


In [ ]:
# ── 2 · CVSS-Weighted Attack Graph ───────────────────────────────────────────

AG = nx.DiGraph()

# Nodes (same as Unit 01)
nodes = {
    'internet':          'external',
    'web_server':        'dmz',
    'api_server':        'dmz',
    'load_balancer':     'dmz',
    'db_primary':        'internal',
    'db_replica':        'internal',
    'admin_workstation': 'internal',
    'dev_laptop':        'internal',
    'log_server':        'internal',
    'backup_server':     'internal',
    'domain_controller': 'critical',
}
AG.add_nodes_from(nodes.keys())

# Edges with CVSS scores
# CVSS 7-10 = critical/high (easy for attacker), 4-6 = medium, 1-3 = low
cvss_edges = [
    ('internet',     'web_server',        9.8),  # public RCE
    ('internet',     'api_server',        8.1),  # auth bypass
    ('web_server',   'load_balancer',     5.3),  # SSRF
    ('web_server',   'api_server',        7.5),  # internal API access
    ('api_server',   'db_primary',        8.8),  # SQL injection → RCE
    ('api_server',   'log_server',        4.3),  # log injection
    ('api_server',   'dev_laptop',        6.5),  # credential exposure
    ('load_balancer','db_primary',        7.2),  # routing bypass
    ('db_primary',   'db_replica',        9.1),  # replication trust
    ('db_primary',   'backup_server',     6.8),  # backup agent trust
    ('db_primary',   'admin_workstation', 5.9),  # DBA session hijack
    ('dev_laptop',   'admin_workstation', 7.8),  # local admin exploit
    ('dev_laptop',   'log_server',        4.1),  # shared logging creds
    ('admin_workstation','domain_controller', 9.0), # domain admin abuse
    ('admin_workstation','log_server',    3.5),  # log service creds
    ('log_server',   'backup_server',     4.6),  # backup API trust
]

for u, v, cvss in cvss_edges:
    AG.add_edge(u, v, cvss=cvss, weight=cvss_to_weight(cvss))

# ── Run Dijkstra from all entry points ────────────────────────────────────────
entry_points = ['internet', 'dev_laptop']
targets      = ['domain_controller', 'db_primary', 'backup_server']

print("MINIMUM-COST ATTACK PATHS (lower cost = easier for attacker)")
print("=" * 70)

all_results = {}
for src in entry_points:
    dist_d, par_d, _ = dijkstra(AG, src)
    all_results[src] = (dist_d, par_d)
    print(f"\nEntry point: {src}")
    print(f"  {'Target':<22} {'Cost':>6}  Path")
    print(f"  {'──────':<22} {'────':>6}  ────")
    for tgt in targets:
        c    = dist_d.get(tgt, float('inf'))
        path = reconstruct_path(par_d, src, tgt)
        if path:
            # Show CVSS scores along path
            path_detail = []
            for i in range(len(path)-1):
                cvss_e = AG[path[i]][path[i+1]]['cvss']
                path_detail.append(f"{path[i]}→{path[i+1]}(CVSS {cvss_e})")
            print(f"  {tgt:<22} {c:>6.2f}  {' | '.join(path_detail)}")
        else:
            print(f"  {tgt:<22} {'∞':>6}  (unreachable)")

# ── Attack cost heatmap ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, src in zip(axes, entry_points):
    dist_d, par_d = all_results[src]
    all_verts = sorted([v for v in AG.nodes() if v != src])
    costs = [dist_d.get(v, float('inf')) for v in all_verts]
    finite_costs = [c for c in costs if c < float('inf')]
    max_c = max(finite_costs) if finite_costs else 1.0

    bar_colors = []
    for v, c in zip(all_verts, costs):
        if c == float('inf'):  bar_colors.append(TS_GRAY)
        elif v in targets:     bar_colors.append(TS_RED if c < 5 else TS_AMBER)
        else:                  bar_colors.append(TS_BLUE)

    bars = ax.barh(range(len(all_verts)),
                   [min(c, max_c*1.1) for c in costs],
                   color=bar_colors, edgecolor='white', height=0.65)

    for i, (v, c) in enumerate(zip(all_verts, costs)):
        label = f"{c:.1f}" if c < float('inf') else "∞"
        ax.text(min(c, max_c*1.1) + 0.1, i, label,
                va='center', fontsize=8, color=TS_GRAY)
        if v in targets:
            ax.text(0.1, i, '▶', va='center', fontsize=9, color=TS_RED)

    ax.set_yticks(range(len(all_verts)))
    ax.set_yticklabels(all_verts, fontsize=9)
    ax.set_xlabel('Minimum attacker cost (lower = easier)')
    ax.set_title(f'Attack costs from: {src}', fontweight='bold', pad=8)

    legend = [mpatches.Patch(color=TS_RED,   label='Target — critical risk'),
              mpatches.Patch(color=TS_AMBER,  label='Target — moderate risk'),
              mpatches.Patch(color=TS_BLUE,   label='Intermediate node'),
              mpatches.Patch(color=TS_GRAY,   label='Unreachable')]
    ax.legend(handles=legend, fontsize=8, loc='lower right')

plt.suptitle('Minimum-Cost Attack Paths (CVSS-Weighted Dijkstra)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

The attack cost bars tell a clear story. From `internet`, the `db_primary` is
cheap to reach — the path goes through high-CVSS vulnerabilities (RCE on the API
server, SQL injection). The `domain_controller` costs more but is still reachable.

From `dev_laptop` (an insider threat or compromised developer machine), the
`domain_controller` is very cheap — the path goes through a local admin exploit
(CVSS 7.8) directly to the admin workstation, then one domain admin abuse step.
**Insider threat is dramatically cheaper than external attack**, a pattern that
Dijkstra makes quantitatively visible.

The bar chart format makes it immediately obvious which targets are dangerously
cheap to reach. Any bar that is short and red is a priority hardening target.


### 3 · Patch Impact Analysis

Not all patches are equal. A patch that removes an edge from the minimum-cost
attack path raises the attack cost directly. A patch on a non-critical-path edge
has no effect on the minimum cost at all.

We compute the **security value** of patching each vulnerability — the increase
in minimum attack cost from entry point to the most critical target — and rank
them to produce a prioritised patching roadmap.


In [ ]:
# ── 3 · Patch Impact Analysis ────────────────────────────────────────────────

primary_target = 'domain_controller'
primary_entry  = 'internet'

# Baseline cost
dist_base, par_base, _ = dijkstra(AG, primary_entry)
baseline_cost = dist_base.get(primary_target, float('inf'))
baseline_path = reconstruct_path(par_base, primary_entry, primary_target)

print(f"Baseline: {primary_entry} → {primary_target}")
print(f"  Cost:  {baseline_cost:.2f}")
print(f"  Path:  {' → '.join(baseline_path)}")
print()

# For each edge, simulate patching it (remove from graph) and re-run Dijkstra
patch_impacts = []
for u, v, data in AG.edges(data=True):
    AG_patched = AG.copy()
    AG_patched.remove_edge(u, v)
    dist_p, _, _ = dijkstra(AG_patched, primary_entry)
    new_cost = dist_p.get(primary_target, float('inf'))
    delta    = new_cost - baseline_cost
    patch_impacts.append({
        'edge':     (u, v),
        'cvss':     data['cvss'],
        'baseline': baseline_cost,
        'new_cost': new_cost,
        'delta':    delta,
        'on_path':  (u, v) in set(zip(baseline_path, baseline_path[1:])),
    })

patch_impacts.sort(key=lambda x: -x['delta'])

print(f"{'Edge':<40} {'CVSS':>5} {'On path':>8} {'Cost Δ':>8}  Impact")
print("─" * 75)
for p in patch_impacts:
    u, v  = p['edge']
    delta_str = (f"+{p['delta']:.2f}" if p['new_cost'] < float('inf')
                 else "→ ∞ (BLOCKS)")
    flag  = '★ CRITICAL PATH' if p['on_path'] else ''
    print(f"  {u}→{v:<28} {p['cvss']:>5.1f} {'YES' if p['on_path'] else 'no':>8} "
          f"{delta_str:>8}  {flag}")

# ── Visualise patch impact bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

labels     = [f"{p['edge'][0]}→{p['edge'][1]}" for p in patch_impacts]
deltas     = [p['delta'] if p['new_cost'] < float('inf') else baseline_cost * 3
              for p in patch_impacts]
bar_colors = [TS_RED   if p['on_path'] else TS_BLUE for p in patch_impacts]

bars = ax.barh(range(len(labels)), deltas, color=bar_colors,
               edgecolor='white', height=0.65)

for i, p in enumerate(patch_impacts):
    label = '→ ∞ (BLOCKS ATTACK)' if p['new_cost'] == float('inf')             else f"+{p['delta']:.2f}"
    ax.text(deltas[i] + 0.05, i, label, va='center', fontsize=8, color=TS_GRAY)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Increase in minimum attack cost (larger = more valuable patch)')
ax.set_title(f'Patch Impact Ranking: internet → domain_controller
'
             f'(Baseline cost = {baseline_cost:.2f}  |  Red = on critical path)',
             pad=10, fontsize=11)

legend = [mpatches.Patch(color=TS_RED,  label='On minimum-cost path — highest priority'),
          mpatches.Patch(color=TS_BLUE, label='Off path — lower priority')]
ax.legend(handles=legend, fontsize=9, loc='lower right')
ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

print(f"\nTop 3 patches by security value:")
for i, p in enumerate(patch_impacts[:3], 1):
    u, v = p['edge']
    print(f"  {i}. Patch {u}→{v} (CVSS {p['cvss']}): "
          f"raises attack cost by {p['delta']:.2f}" if p['new_cost'] < float('inf')
          else f"  {i}. Patch {u}→{v} (CVSS {p['cvss']}): COMPLETELY BLOCKS ATTACK PATH")


**What do you see?**

The patch impact chart transforms a vulnerability list into an **ordered remediation
roadmap**. The red bars are on the minimum-cost attack path — patching any of them
directly raises the attacker's cost. The blue bars are off the critical path —
patching them has zero effect on the minimum-cost route (the attacker simply uses
a different path).

The entries marked "→ ∞ (BLOCKS ATTACK)" are the most valuable: patching that
single edge removes all paths from the entry point to the target. These are the
single-point-of-failure edges in the attack graph — the defender's highest-return
investments.

**The formal implication.** Dijkstra-based patch impact analysis converts the
question "which vulnerabilities should we patch first?" from a subjective risk
discussion into a precise mathematical answer: patch the edges on the minimum-cost
path, in order of the cost delta they produce. Everything else is a lower priority
until those are resolved.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. K SHORTEST PATHS
#    An attacker who is blocked from the cheapest path will try the second-cheapest,
#    then the third, and so on.
#    NetworkX has nx.shortest_simple_paths(G, source, target, weight='weight').
#    Find the 5 cheapest paths from 'internet' to 'domain_controller'.
#    How much more expensive is path #2 than path #1? Path #5 than #1?
#    What does this tell you about the attacker's fallback options?
#
# 2. MULTI-TARGET COST
#    The attacker wants to reach ANY target, not just one.
#    Define: attack_cost(G, s) = min over all targets t of delta_w(s, t)
#    Compute this for both entry points.
#    Which entry point gives the cheapest first target access?
#    Which target is reached most cheaply across all entry points?
#
# 3. DEFENDER BUDGET OPTIMISATION
#    Suppose the defender has a "patch budget" of 3 edges they can remove.
#    Which 3 edges should they remove to maximise the minimum attack cost
#    to the domain_controller from the internet?
#    (Hint: greedy — remove the highest-delta edge, recompute, repeat.)
#    Is the greedy choice always optimal? Can you think of a case where it isn't?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Weighted shortest path** | Min $\sum w$ over all paths | Minimum attacker effort |
| **Dijkstra's algorithm** | Greedy relaxation with priority queue | Computes minimum cost to every vertex |
| **Relaxation** | Update dist if cheaper route found | Discovering the best attack route |
| **CVSS → weight** | $w = 10 - \text{CVSS} + 0.1$ | Low weight = easy to exploit |
| **Critical path** | Edges on the minimum-cost path | Defender's priority patching list |
| **Patch delta** | Cost increase from removing an edge | Security value of a patch |
| **$\delta_w(s,t) = \infty$** | No path exists | Isolation guarantee — Unit 03 |

---

## Up Next

**Module 05 · Unit 03 — Zero-Trust Segmentation**

We close the module by formalising the defender's side. Zero-trust segmentation
is the systematic removal of edges from the attack graph to raise the minimum
attack cost above the attacker's threshold — or to make it infinite (no path).
We prove the fundamental theorem: removing any edge $(u,v)$ from the minimum-cost
path strictly increases $\delta_w(s,t)$. Then we build a segmentation strategy
that achieves maximum cost increase with minimum network disruption.

$\rightarrow$ `modules/module-05/unit-03-zero-trust-segmentation.ipynb`
